# 03. 공식 저해상도 training split에서 네 baseline 학습

## 1. 설정
공통 변경은 00, 이 단계만 변경하려면 아래 변수를 편집하세요. CosFace 추가는 아래 HEADS/HEAD_PARAMETERS에 반영되어 있어 00·02를 다시 실행할 필요가 없습니다. 02 학습은 완료까지 유지하고, 03은 별도 커널에서 실행하세요. 공통 부모를 독립 복사해 각 head와 backbone을 함께 학습합니다.

In [1]:
from pathlib import Path
import sys, json
BASE_DIR = Path("F:/FaceX-Zoo")
SETTINGS_PATH = BASE_DIR / "fine-tune/data/notebook_settings.json"
for path in (BASE_DIR, BASE_DIR / "fine-tune"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from modules.notebook_workflow import resolve_settings, save_settings, plan_matrix
from modules.finetune_protocol import default_profiles
saved = resolve_settings(BASE_DIR, SETTINGS_PATH, require_saved=True)
SEEDS = list(saved["project"]["seed_list"])
HEADS = ["arcface", "adaface", "magface", "cosface"]  # 03/04에서 같은 목록 사용; 00 저장값보다 우선
PARENT_SEED = saved["campaign"]["parent_seed"]
PRETRAIN_EPOCHS = saved["campaign"]["pretrain_epochs"]
PARENT_CHECKPOINT = saved["campaign"]["parent_checkpoint"]
PARENT_SHA256 = saved["campaign"]["parent_sha256"]
FINETUNE_DATASETS = list(saved["campaign"]["finetune_datasets"])
FINETUNE_PROFILES = saved["finetune_profiles"]
AUDIT_WORKERS = 4  # 전체 JPEG decode는 CPU에서 수행; 기존 검증은 snapshot/hash 확인 후 재사용
KEEP_RAW_RESULTS = saved["project"]["keep_raw_results"]
NUM_WORKERS = saved["training"]["num_workers"]
PREFETCH_FACTOR = saved["training"]["prefetch_factor"]
PERSISTENT_WORKERS = saved["training"]["persistent_workers"]
TORCH_THREADS = saved["training"]["torch_threads"]
LOSS_SYNC_BATCHES = saved["training"].get("loss_sync_batches", 20)
RECOVERY_MINUTES = saved["training"]["recovery_minutes"]
DEVICE = saved["project"]["device"]
DATASET = dict(saved["dataset"])  # 02는 VGGFace2, 03은 선택한 대상 profile을 사용
BACKBONE = dict(saved["backbone"])
TRAINING = dict(saved["training"])
HEAD_PARAMETERS = saved["heads"]
HEAD_PARAMETERS["cosface"].update(type="CosFace", margin=0.3, scale=64.0)  # 논문 IV-A-2 저해상도 설정
PATHS = dict(saved["paths"])
REUSE_COMPLETED = saved["campaign"]["reuse_completed"]
EXECUTE = True  # 학습하려면 명시적으로 True 설정
STAGE = "finetune"
RUN_SMOKE_TEST = False






# RESUME_CHECKPOINTS = {}  # run_id -> 정확한 재개 checkpoint 경로


RESUME_CHECKPOINTS = {
    "finetune_survface_magface_seed8975": (
        "F:/FaceX-Zoo/fine-tune/checkpoints/shared_parent_v3/"
        "finetune_survface_magface_seed8975/"
        "finetune_mobilefacenet_1024d_magface_epoch_20.pt"
    )
}




DATASET.update(train_archive=str(Path(DATASET["raw_root"])/"data/vggface2_train.tar.gz"),
 test_archive=str(Path(DATASET["raw_root"])/"data/vggface2_test.tar.gz"),
 meta_csv=str(Path(DATASET["raw_root"])/"meta/identity_meta.csv"))
settings = resolve_settings(BASE_DIR, SETTINGS_PATH, overrides={
 "project":dict(seed=SEEDS[0],seed_list=SEEDS,keep_raw_results=KEEP_RAW_RESULTS,device=DEVICE),
 "campaign":dict(heads=HEADS,reuse_completed=REUSE_COMPLETED,parent_seed=PARENT_SEED,
    pretrain_epochs=PRETRAIN_EPOCHS,parent_checkpoint=PARENT_CHECKPOINT,parent_sha256=PARENT_SHA256,
    finetune_datasets=FINETUNE_DATASETS),
 "finetune_profiles":FINETUNE_PROFILES,"dataset":DATASET,"backbone":BACKBONE,"heads":HEAD_PARAMETERS,
 "training":{**TRAINING,"num_workers":NUM_WORKERS,"prefetch_factor":PREFETCH_FACTOR,
    "persistent_workers":PERSISTENT_WORKERS,"torch_threads":TORCH_THREADS,"recovery_minutes":RECOVERY_MINUTES,"loss_sync_batches":LOSS_SYNC_BATCHES},
 "paths":PATHS})


## 2. 계획

In [2]:
plan = plan_matrix(settings, STAGE)
for row in plan:
    print(row)
print("예정:",len(plan),"EXECUTE:",EXECUTE,"SMOKE:",RUN_SMOKE_TEST)

{'stage': 'finetune', 'head': 'arcface', 'seed': 8972, 'dataset': 'tinyface', 'run_id': 'finetune_tinyface_arcface_seed8972'}
{'stage': 'finetune', 'head': 'adaface', 'seed': 8972, 'dataset': 'tinyface', 'run_id': 'finetune_tinyface_adaface_seed8972'}
{'stage': 'finetune', 'head': 'magface', 'seed': 8972, 'dataset': 'tinyface', 'run_id': 'finetune_tinyface_magface_seed8972'}
{'stage': 'finetune', 'head': 'cosface', 'seed': 8972, 'dataset': 'tinyface', 'run_id': 'finetune_tinyface_cosface_seed8972'}
{'stage': 'finetune', 'head': 'arcface', 'seed': 8973, 'dataset': 'tinyface', 'run_id': 'finetune_tinyface_arcface_seed8973'}
{'stage': 'finetune', 'head': 'adaface', 'seed': 8973, 'dataset': 'tinyface', 'run_id': 'finetune_tinyface_adaface_seed8973'}
{'stage': 'finetune', 'head': 'magface', 'seed': 8973, 'dataset': 'tinyface', 'run_id': 'finetune_tinyface_magface_seed8973'}
{'stage': 'finetune', 'head': 'cosface', 'seed': 8973, 'dataset': 'tinyface', 'run_id': 'finetune_tinyface_cosface_see

## 3. 재개 정책

In [3]:
print("공통 부모:", PARENT_CHECKPOINT or "02의 PARENT_SEED 고정 경로", "parent seed:", PARENT_SEED)
print("시작 전 00 설정 저장과 01 데이터 준비를 확인하세요.")
from modules.finetune_protocol import prepare_selected_protocols
if EXECUTE:
    for protocol in prepare_selected_protocols(settings, audit_workers=AUDIT_WORKERS):
        print(protocol)
else:
    print("계획 모드: 공식 split 및 전체 이미지 검증/목록 생성 미실행")

공통 부모: 02의 PARENT_SEED 고정 경로 parent seed: 8972
시작 전 00 설정 저장과 01 데이터 준비를 확인하세요.
{'name': 'tinyface', 'raw_root': 'F:\\FaceX-Zoo\\tinyface', 'num_classes': 2570, 'images': 7804, 'train_list': 'F:\\FaceX-Zoo\\fine-tune\\data\\protocols\\local_qmul_v1\\tinyface\\train_list.txt', 'train_sha256': 'fdd3c1d2a8c3d386cafacc9b3a78e7ce23896542fbf900ae750cf0ea48f48756', 'training_manifest': 'F:\\FaceX-Zoo\\fine-tune\\data\\protocols\\local_qmul_v1\\tinyface\\training_manifest.csv', 'training_manifest_sha256': '307d009a75d31f056ee9cd2ccb360f54da5a53f9339bf93faf511e504a12b4c3', 'evaluation_manifest': 'F:\\FaceX-Zoo\\fine-tune\\data\\protocols\\local_qmul_v1\\tinyface\\evaluation_manifest.csv', 'evaluation_manifest_sha256': '58e7f4de90266b19bc9e53532883ce4d79da6e9cdf7c6f200fcbb2ba6a9ffa98', 'identity_disjoint': True, 'evaluation_identities': 2569, 'identity_check_scope': 'native IDs verified where available; official distractor identity disjointness relies on dataset protocol', 'evaluation_images_wit

## 4. 별도 프로세스 실행
노트북 Interrupt는 stop 파일을 만들고 학습은 optimizer step 완료 후 재개 checkpoint를 저장합니다. Kernel 강제 종료는 이 절차를 실행하지 않을 수 있으므로 출력된 stop 파일 경로 또는 progress.json을 확인하세요. 입력 처리만 최적화하며 baseline batch 64·Adam·40 epoch는 유지합니다.

In [4]:
from modules.process_runner import launch_campaign
results = []
if EXECUTE:
    results=launch_campaign(BASE_DIR,settings,STAGE,RESUME_CHECKPOINTS,smoke=RUN_SMOKE_TEST)
else:
    print("계획 또는 benchmark 모드: 정식 작업 미실행")

Child PID 42716; log: F:\FaceX-Zoo\fine-tune\data\launches\7ddedaed4b4f4991b4f7bdfe43638941\console.log; stop request: F:\FaceX-Zoo\fine-tune\data\launches\7ddedaed4b4f4991b4f7bdfe43638941\stop.request
Device and runtime checks precede execution; PID=42716
[1/40] finetune_tinyface_arcface_seed8972
2026-10-18 14:55:20,347 INFO Reproducibility seed set to: 8972
2026-10-18 14:55:20,642 INFO Verified parent checkpoint: F:\FaceX-Zoo\fine-tune\checkpoints\shared_parent_v3\pretrain_arcface_seed8972\pretrain_mobilefacenet_1024d.pt (SHA-256: 742c59c294895ee2058e835e8985ee35b5589ad5d4d07cdaee96a8f40ab35fe1)
2026-10-18 14:55:20,642 INFO Parent checkpoint SHA-256 integrity check PASSED.
2026-10-18 14:55:20,779 INFO Built FaceRecognitionModel: Backbone=MobileFaceNet(dim=1024, dropout=0.6), Head=ArcFace(num_class=2570)
2026-10-18 14:55:23,095 INFO Loaded backbone state_dict.
2026-10-18 14:55:23,097 INFO Loaded head state_dict.
2026-10-18 14:55:23,144 INFO Loaded optimizer state_dict.
2026-10-18 14:5

## 5. 완료 상태

In [5]:
for result in results:
    print(result)
print("정식 완료:",sum(r.get("completed") is True for r in results),"/",len(plan))
print("실패 시 summary에 미실행 범위를 기록하고 후속 작업을 중단합니다.")
print("해석 안내:",BASE_DIR/"fine-tune/README.md")

{'stage': 'finetune', 'head': 'arcface', 'seed': 8972, 'dataset': 'tinyface', 'run_id': 'finetune_tinyface_arcface_seed8972', 'status': 'reused_completed', 'completed': True, 'checkpoint': 'F:\\FaceX-Zoo\\fine-tune\\checkpoints\\shared_parent_v3\\finetune_tinyface_arcface_seed8972\\finetune_mobilefacenet_1024d_arcface_epoch_40.pt', 'sha256': 'b8a93b9ed1e1ab3977f3734f9b1bd9d0771b2f56106da5f5fdb15db3423ae8f9', 'config_hash': 'b400e1d520dbc3c5bf73da9267d8431998f08d2843f14df8597c88f8f1082ff0', 'parent_hash': '742c59c294895ee2058e835e8985ee35b5589ad5d4d07cdaee96a8f40ab35fe1', 'keep_raw_results': True, 'requested_keep_raw_results': True, 'device': 'cuda:0', 'gpu': 'NVIDIA GeForce GTX 1080 Ti'}
{'stage': 'finetune', 'head': 'adaface', 'seed': 8972, 'dataset': 'tinyface', 'run_id': 'finetune_tinyface_adaface_seed8972', 'status': 'reused_completed', 'completed': True, 'checkpoint': 'F:\\FaceX-Zoo\\fine-tune\\checkpoints\\shared_parent_v3\\finetune_tinyface_adaface_seed8972\\finetune_mobilefacen